# Baseline v6.1 — Retrain CE với Hybrid Candidates

**Vấn đề v6:** CE được train với Dense candidates, nhận Hybrid candidates → mismatch  
**Fix v6.1:** Mine hard negatives từ **Hybrid RRF** thay vì chỉ Dense

```
Hard Neg Mining v6.1:
  BM25(query, top-30) ─┐
                       ├→ RRF merge → skip top-14 → lấy rank 15-30 làm HN
  Dense(query, top-30) ─┘
```

## Cell 0 — Config & Imports

In [ ]:
import json, csv, time, random, gc, re
import numpy as np, faiss, torch
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import DataLoader
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator

ROOT, DATA_DIR = Path("."), Path(".") / "data"
EVAL_DIR = ROOT / "outputs" / "eval"
TMP_DIR  = ROOT / "outputs" / "tmp"
MDL_DIR  = ROOT / "outputs" / "models"

TRAIN_FILE   = DATA_DIR / "train.jsonl"
DEV_FILE     = DATA_DIR / "dev.jsonl"
TRAIN_NEG    = DATA_DIR / "train_with_neg.jsonl"
EVAL_QA_FILE = EVAL_DIR / "eval_qa.jsonl"
FT_BI_PATH   = MDL_DIR  / "legal_hf_finetuned" / "final"
FAISS_V4     = TMP_DIR  / "faiss_v4.index"
MAP_V4       = TMP_DIR  / "faiss_mapping_v4.jsonl"
RERANK_CSV_V5= EVAL_DIR / "rerank_metrics_v5.csv"
RERANK_CSV_V61=EVAL_DIR / "rerank_metrics_v6_1.csv"
CE_V61_DIR   = MDL_DIR  / "cross_encoder_v6_1"

BASE_CE_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
CE_EPOCHS     = 5
CE_BATCH      = 32
CE_MAX_LEN    = 256
SEED          = 42
SKIP_TOP_K    = 14    # giữ nguyên skip=14 (best từ D_skip14)
TOP_MINE      = 30
HARD_NEG_PER  = 2
RRF_K         = 60
TOP_N_EVAL    = 50

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED)
CE_V61_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE} | SKIP_TOP_K={SKIP_TOP_K} | Mining from HYBRID RRF rank {SKIP_TOP_K+1}-{TOP_MINE}")

## Cell 1 — Utilities

In [ ]:
def load_jsonl(path):
    rows, err = [], 0
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try: rows.append(json.loads(line))
            except: err += 1
    if err: print(f"  ⚠ {err} errors")
    return rows

def is_hit(corpus_id, ec, corpus):
    row = corpus[corpus_id]
    for e in ec:
        ci = e.get("chunk_index",-2)
        if ci!=-1 and row["chunk_index"]==ci: return True
        if row["van_ban"]==e.get("van_ban","") and row["dieu"]==e.get("dieu","") and row["khoan"]==e.get("khoan",""): return True
    return False

def is_pos_meta(cand, meta):
    if cand.get("van_ban","")==meta.get("van_ban","") and cand.get("van_ban","")!="" \
       and cand.get("dieu","")==meta.get("dieu","") and cand.get("khoan","")==meta.get("khoan",""): return True
    ci = meta.get("chunk_index",-2)
    return ci!=-1 and cand.get("chunk_index")==ci

def tokenize_vi(text):
    text = text.lower()
    tokens = re.findall(r'[a-záàảãạăắằẳẵặâấầẩẫậéèẻẽẹêếềểễệíìỉĩịóòỏõọôốồổỗộơớờởỡợúùủũụưứừửữựýỳỷỹỵđ0-9]+', text)
    return tokens if tokens else text.split()

def rrf_merge(ids_a, ids_b, k=60):
    scores = {}
    for rank, i in enumerate(ids_a, 1): scores[i] = scores.get(i,0) + 1.0/(k+rank)
    for rank, i in enumerate(ids_b, 1): scores[i] = scores.get(i,0) + 1.0/(k+rank)
    return [i for i,_ in sorted(scores.items(), key=lambda x:x[1], reverse=True)]

def avg(lst): return round(sum(lst)/len(lst),4) if lst else 0.0
print("Utilities ✓")

## Cell 2 — Build Corpus + BM25 + Load v4 FAISS

In [ ]:
# Corpus
seen_passages = {}
for f in [TRAIN_FILE, DEV_FILE, TRAIN_NEG]:
    for r in load_jsonl(f):
        p = r.get("passage","")
        if p and p not in seen_passages:
            meta = r.get("meta",{})
            seen_passages[p] = {"passage":p,
                "chunk_index":meta.get("chunk_index",-1), "van_ban":meta.get("van_ban",""),
                "chuong":meta.get("chuong",""), "dieu":meta.get("dieu",""),
                "khoan":meta.get("khoan",""), "diem":meta.get("diem","")}
corpus = list(seen_passages.values())
print(f"Corpus: {len(corpus)} passages")

# BM25
print("Building BM25...")
tokenized = [tokenize_vi(c["passage"]) for c in tqdm(corpus, desc="Tokenize", leave=False)]
bm25 = BM25Okapi(tokenized)
print("BM25 ✓")

# FAISS v4
print("Loading FAISS v4 + bi-encoder...")
ft_bi      = SentenceTransformer(str(FT_BI_PATH), device=DEVICE)
index_v4   = faiss.read_index(str(FAISS_V4))
mapping_v4 = load_jsonl(MAP_V4)
# faiss_id → corpus_id
p2cid = {c["passage"]:i for i,c in enumerate(corpus)}
f2cid = {e["faiss_id"]: p2cid[e["passage"]] for e in mapping_v4 if e["passage"] in p2cid}
print(f"FAISS ✓ | {index_v4.ntotal} vectors | f2cid: {len(f2cid)}")

## Cell 3 — Mine Hard Negatives từ Hybrid RRF

In [ ]:
pos_rows = [r for r in load_jsonl(TRAIN_NEG) if r.get("label")==1]
random.seed(SEED); random.shuffle(pos_rows)
print(f"Positive rows: {len(pos_rows)}")
print(f"Mining HN from HYBRID (BM25+Dense) rank {SKIP_TOP_K+1}-{TOP_MINE}")

train_data = []
stats = {"pos":0,"neg":0,"no_neg":0}

for r in tqdm(pos_rows, desc="Mine Hybrid HN"):
    query = r.get("query","").strip()
    pos_p = r.get("passage","").strip()
    meta  = r.get("meta",{})
    if not query or not pos_p: continue

    train_data.append(InputExample(texts=[query,pos_p], label=1.0))
    stats["pos"] += 1

    # BM25 retrieve
    bm25_scores = bm25.get_scores(tokenize_vi(query))
    bm25_ids    = np.argsort(bm25_scores)[::-1][:TOP_MINE].tolist()

    # Dense retrieve
    q_emb = ft_bi.encode([query], normalize_embeddings=True,
                          convert_to_numpy=True).astype("float32")
    _, fids = index_v4.search(q_emb, TOP_MINE)
    dense_cids = [f2cid[fid] for fid in fids[0].tolist() if fid>=0 and fid in f2cid]

    # Hybrid RRF merge
    hybrid_ids = rrf_merge(bm25_ids, dense_cids, k=RRF_K)

    # Skip top-SKIP_TOP_K, lấy rank SKIP_TOP_K+1 → TOP_MINE
    added = 0
    for cid in hybrid_ids[SKIP_TOP_K:]:
        if cid < 0 or cid >= len(corpus) or added >= HARD_NEG_PER: break
        cand = corpus[cid]
        if cand["passage"]==pos_p or is_pos_meta(cand, meta): continue
        train_data.append(InputExample(texts=[query,cand["passage"]], label=0.0))
        added += 1; stats["neg"] += 1
    if added==0: stats["no_neg"] += 1

print(f"  pos={stats['pos']}, neg={stats['neg']} ({stats['neg']/max(stats['pos'],1):.1f}/q)")
print(f"  Total: {len(train_data)} | pos_ratio: {stats['pos']/len(train_data):.0%}")

# Dev set
dev_rows = load_jsonl(DEV_FILE)
random.seed(SEED); random.shuffle(dev_rows)
dev_data = []
for r in tqdm(dev_rows[:500], desc="Dev HN", leave=False):
    query = r.get("query","").strip(); pos_p = r.get("passage","").strip()
    meta  = r.get("meta",{})
    if not query or not pos_p: continue
    dev_data.append(InputExample(texts=[query,pos_p], label=1.0))
    bm25_ids   = np.argsort(bm25.get_scores(tokenize_vi(query)))[::-1][:TOP_MINE].tolist()
    q_emb      = ft_bi.encode([query], normalize_embeddings=True,
                               convert_to_numpy=True).astype("float32")
    _, fids    = index_v4.search(q_emb, TOP_MINE)
    dense_cids = [f2cid[fid] for fid in fids[0].tolist() if fid>=0 and fid in f2cid]
    hybrid_ids = rrf_merge(bm25_ids, dense_cids, k=RRF_K)
    for cid in hybrid_ids[SKIP_TOP_K:]:
        if cid < 0 or cid >= len(corpus): break
        cand = corpus[cid]
        if cand["passage"]==pos_p or is_pos_meta(cand, meta): continue
        dev_data.append(InputExample(texts=[query,cand["passage"]], label=0.0))
        break
print(f"Dev: {len(dev_data)} samples")

## Cell 4 — Train CE v6.1
> ⏱️ ~25-35 phút (RTX 3050 Ti, 5 epochs)

In [ ]:
# Giải phóng bi-encoder VRAM
del ft_bi; gc.collect()
torch.cuda.empty_cache() if DEVICE=="cuda" else None
print("VRAM cleared ✓")

random.seed(SEED); random.shuffle(train_data)
ce = CrossEncoder(BASE_CE_MODEL, num_labels=1, max_length=CE_MAX_LEN, device=DEVICE)
evaluator = CEBinaryClassificationEvaluator.from_input_examples(dev_data, name="v6_1")
warmup    = int(len(train_data)/CE_BATCH * CE_EPOCHS * 0.1)
print(f"Train: {len(train_data)} | Dev: {len(dev_data)} | Epochs: {CE_EPOCHS} | Warmup: {warmup}")

t0 = time.time()
ce.fit(
    train_dataloader=DataLoader(train_data, shuffle=True, batch_size=CE_BATCH),
    evaluator=evaluator, epochs=CE_EPOCHS,
    warmup_steps=warmup, output_path=str(CE_V61_DIR),
    use_amp=(DEVICE=="cuda"),
)
elapsed = round((time.time()-t0)/60,1)
print(f"Training done in {elapsed} min")
saved = CE_V61_DIR / "saved_model"
ce.save(str(saved))
print(f"Saved → {saved}")

## Cell 5 — Evaluate v6.1: Hybrid + CE (trained on hybrid HN)

In [ ]:
gc.collect(); torch.cuda.empty_cache() if DEVICE=="cuda" else None

print("Reloading v4 bi-encoder + FAISS...")
ft_bi      = SentenceTransformer(str(FT_BI_PATH), device=DEVICE)
index_v4   = faiss.read_index(str(FAISS_V4))
mapping_v4 = load_jsonl(MAP_V4)
f2cid_eval = {e["faiss_id"]: p2cid[e["passage"]] for e in mapping_v4 if e["passage"] in p2cid}
eval_qa    = load_jsonl(EVAL_QA_FILE)
print(f"Loaded ✓ | {len(eval_qa)} questions")

variants = {
    "Dense_only":  {"R@1":[],"R@3":[],"R@5":[],"MRR@10":[]},
    "Hybrid_RRF":  {"R@1":[],"R@3":[],"R@5":[],"MRR@10":[]},
    "Hybrid+CE61": {"R@1":[],"R@3":[],"R@5":[],"MRR@10":[]},
}

def eval_ranks(ids_, ec, res):
    for k,key in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
        res[key].append(1 if any(is_hit(i,ec,corpus) for i in ids_[:k]) else 0)
    mrr=0.0
    for rank,i in enumerate(ids_[:10],1):
        if is_hit(i,ec,corpus): mrr=1.0/rank; break
    res["MRR@10"].append(mrr)

for item in tqdm(eval_qa, desc="Evaluate v6.1"):
    query=item["query"]; ec=item["expected_citations"]

    # Dense
    q_emb = ft_bi.encode([query], normalize_embeddings=True,
                          convert_to_numpy=True).astype("float32")
    _, fids = index_v4.search(q_emb, TOP_N_EVAL)
    dense_cids = [f2cid_eval[fid] for fid in fids[0].tolist() if fid>=0 and fid in f2cid_eval]
    eval_ranks(dense_cids[:TOP_N_EVAL], ec, variants["Dense_only"])

    # Hybrid RRF
    bm25_ids   = np.argsort(bm25.get_scores(tokenize_vi(query)))[::-1][:100].tolist()
    q_emb2     = ft_bi.encode([query], normalize_embeddings=True,
                               convert_to_numpy=True).astype("float32")
    _, fids2   = index_v4.search(q_emb2, 100)
    d_cids2    = [f2cid_eval[fid] for fid in fids2[0].tolist() if fid>=0 and fid in f2cid_eval]
    hybrid_ids = rrf_merge(bm25_ids, d_cids2, k=RRF_K)[:TOP_N_EVAL]
    eval_ranks(hybrid_ids, ec, variants["Hybrid_RRF"])

    # Hybrid + CE v6.1
    cands  = [(corpus[i]["passage"],i) for i in hybrid_ids if i<len(corpus)]
    rscore = ce.predict([[query,c[0]] for c in cands], batch_size=32) if cands else []
    ranked = sorted(zip(rscore,[c[1] for c in cands]),reverse=True)
    eval_ranks([x[1] for x in ranked], ec, variants["Hybrid+CE61"])

# Results
V5  = {"R@1":0.5418,"R@3":0.6873,"R@5":0.7245,"MRR@10":0.6307}
V61 = {m:avg(variants["Hybrid+CE61"][m]) for m in ["R@1","R@3","R@5","MRR@10"]}

print("\n" + "="*80)
print(f"  {'Metric':<10} {'v5 (D_skip14)':>16} {'v6.1 (H+CE61)':>16} {'Δ':>10} {'Win':>5}")
print("="*80)
for m in ["R@1","R@3","R@5","MRR@10"]:
    d = V61[m]-V5[m]
    win = "✅" if d>0.001 else ("❌" if d<-0.001 else "=")
    print(f"  {m:<10} {V5[m]:>16.4f} {V61[m]:>16.4f} {d:>+9.4f} {win:>5}")
print("="*80)

print("\n── Breakdown ──")
print(f"  {'Variant':<14} {'R@1':>8} {'R@3':>8} {'R@5':>8} {'MRR':>8}")
print("  "+"-"*46)
for vn,vr in variants.items():
    marker = " ★" if vn=="Hybrid+CE61" else ""
    print(f"  {vn:<14} {avg(vr['R@1']):>8.4f} {avg(vr['R@3']):>8.4f} {avg(vr['R@5']):>8.4f} {avg(vr['MRR@10']):>8.4f}{marker}")
print(f"  {'v5 (Dense+CE)':14} {V5['R@1']:>8.4f} {V5['R@3']:>8.4f} {V5['R@5']:>8.4f} {V5['MRR@10']:>8.4f}")

## Cell 6 — Lưu CSV

In [ ]:
rows = []
for m in ["R@1","R@3","R@5","MRR@10"]:
    rows.append({"metric":m,
        "v5_D_skip14": V5[m],
        "v6_1_dense_only": avg(variants["Dense_only"][m]),
        "v6_1_hybrid_rrf": avg(variants["Hybrid_RRF"][m]),
        "v6_1_hybrid_ce61": V61[m],
    })
with open(RERANK_CSV_V61,"w",newline="",encoding="utf-8") as f:
    w = csv.DictWriter(f,fieldnames=["metric","v5_D_skip14","v6_1_dense_only","v6_1_hybrid_rrf","v6_1_hybrid_ce61"])
    w.writeheader(); w.writerows(rows)
print(f"Saved → {RERANK_CSV_V61} ✓")